In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Library

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Flatten, Dense, Embedding, Dropout, Concatenate, MultiHeadAttention, Lambda
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

In [ ]:
def data_split(df, test_size=0.2, valid_size=0.125, random_state=42):

    # 1차 분할: train+valid, test
    df_train, df_test = train_test_split(df, test_size=test_size, random_state=random_state)

    # 2차 분할: train, valid
    df_train, df_valid = train_test_split(df_train, test_size=valid_size, random_state=random_state)

    df_train = df_train.reset_index(drop=True)
    df_valid = df_valid.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)

    'Training Dataset'
    user_id_train = df_train['user_id'].values
    item_id_train = df_train['asin'].values
    bert_train = np.array(df_train['bert_emb'].tolist())
    vit_train = np.array(df_train['ViT_emb_768'].tolist())
    y_train = df_train['rating'].values

    'Valid Dataset'
    user_id_valid = df_valid['user_id'].values
    item_id_valid = df_valid['asin'].values
    bert_valid = np.array(df_valid['bert_emb'].tolist())
    vit_valid = np.array(df_valid['ViT_emb_768'].tolist())
    y_valid = df_valid['rating'].values

    'Test Dataset'
    user_id_test = df_test['user_id'].values
    item_id_test = df_test['asin'].values
    bert_test = np.array(df_test['bert_emb'].tolist())
    vit_test = np.array(df_test['ViT_emb_768'].tolist())
    y_test = df_test['rating'].values

    return (
        user_id_train, item_id_train, bert_train, vit_train, y_train,
        user_id_valid, item_id_valid, bert_valid, vit_valid, y_valid,
        user_id_test, item_id_test, bert_test, vit_test, y_test
    )

# Load Data

In [3]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Cell_Phones_RQ_v3.pkl')

print(df.shape)
df.head()

(23491, 8)


In [4]:
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

# 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
print(n_users)
n_items = df['asin'].nunique()
print(n_items)

4149
4975


In [ ]:
user_id_train, item_id_train, bert_train, vit_train, y_train, \
user_id_valid, item_id_valid, bert_valid, vit_valid, y_valid, \
user_id_test, item_id_test, bert_test, vit_test, y_test = data_split(df)

# Model

In [ ]:
def Proposed_RQ_1(num_users, num_items, dropout, learning_rate):

    # 사용자 ID
    user_id_input = Input(shape=(1,), name='user_id_input')
    user_emb = Embedding(input_dim=num_users, output_dim=64, name='user_embedding')(user_id_input)

    # 아이템 ID
    item_id_input = Input(shape=(1,), name='item_id_input')
    item_emb = Embedding(input_dim=num_items, output_dim=64, name='item_embedding')(item_id_input)

    # Text Input
    bert_input = Input(shape=(768,), name='roberta_input')

    # Image Input
    vit_input = Input(shape=(768,), name='vit_input')

    # Text MLP
    bert_nlp = Dense(512, activation='relu')(bert_input)
    bert_nlp = Dropout(dropout)(bert_nlp)
    bert_nlp = Dense(256, activation='relu')(bert_nlp)
    bert_nlp = Dropout(dropout)(bert_nlp)                            # (B, 256)

    # Image MLP
    vit_nlp = Dense(512, activation='relu')(vit_input)
    vit_nlp = Dropout(dropout)(vit_nlp)
    vit_nlp = Dense(256, activation='relu')(vit_nlp)
    vit_nlp = Dropout(dropout)(vit_nlp)                              # (B, 256)

    # seq_len=1 축 추가 (모두 (B, 1, D)로 정렬)
    bert_vec = Lambda(lambda x: tf.expand_dims(x, axis=1), name='roberta_expand')(bert_nlp)  # (B,1,256)
    vit_vec  = Lambda(lambda x: tf.expand_dims(x, axis=1), name='vit_expand')(vit_nlp)       # (B,1,256)

    # Concat: (B,1,320)로 맞춤
    user_rep = Concatenate(axis=-1, name='user_rep_cat')([user_emb, bert_vec])   # (B,1,64+256)
    item_rep = Concatenate(axis=-1, name='item_rep_cat')([item_emb, vit_vec])    # (B,1,256+64)

    # Co-Attention(bi-directional cross attention)
    text_attention = MultiHeadAttention(
        num_heads=8, key_dim=64, name='bert_cross_attention'
    )(query=user_rep, value=item_rep, key=item_rep)                               # (B,1,256)

    img_attention = MultiHeadAttention(
        num_heads=8, key_dim=64, name='vit_cross_attention'
    )(query=item_rep, value=user_rep, key=user_rep)                               # (B,1,256)

    # seq 차원 제거 -> (B, 256)
    text_attention = Lambda(lambda x: tf.squeeze(x, axis=1), name='bert_squeeze')(text_attention)
    img_attention  = Lambda(lambda x: tf.squeeze(x, axis=1),  name='vit_squeeze')(img_attention)

    # 결합 -> (B, 512)
    multi_modal_rep = Concatenate(axis=-1, name='mm_concat')([text_attention, img_attention])

    # Co-Attention MLP
    multi_modal_rep = Dense(256, activation='relu')(multi_modal_rep)
    multi_modal_rep = Dropout(dropout)(multi_modal_rep)

    # Final
    final = Dense(1, activation='linear', name='rating_prediction')(multi_modal_rep)

    model = Model(
        inputs=[user_id_input, item_id_input, bert_input, vit_input],
        outputs=final
    )
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mean_squared_error',
        metrics=['mean_absolute_error', 'mean_squared_error']
    )

    return model

In [ ]:
dropout = 0.0
learning_rate = 0.001

In [8]:
model = Proposed_RQ_1(n_users, n_items, dropout, learning_rate)
model.summary()

Model: "functional"

 Total params: 3,112,833 (11.87 MB)

 Trainable params: 3,112,833 (11.87 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stopping = EarlyStopping(monitor ='val_loss', min_delta = 0.001, patience = 5, verbose = 1, mode = 'min', restore_best_weights = True)

In [ ]:
history = model.fit([user_id_train, item_id_train, bert_train, vit_train],
                    y_train,
                    validation_data = ([user_id_valid, item_id_valid, bert_valid, vit_valid], y_valid),
                    epochs = 100,
                    batch_size = 128,
                    callbacks=[early_stopping])

In [11]:
# 모델 성능 측정
predicted_ratings = model.predict([user_id_test, item_id_test, bert_test, vit_test])

MAE = mean_absolute_error(y_test, predicted_ratings)
MSE = mean_squared_error(y_test, predicted_ratings)
RMSE = np.sqrt(MSE)
MAPE = mean_absolute_percentage_error(y_test, predicted_ratings) * 100


print(f'MAE: {MAE:.4f}')
print(f'MSE: {MSE:.4f}')
print(f'RMSE: {RMSE:.4f}')
print(f'MAPE: {MAPE:.4f}%')

147/147 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step
MAE: 0.4410
MSE: 0.4116
RMSE: 0.6416
MAPE: 13.6988%
